In [1]:
import warnings
warnings.simplefilter('ignore')

In [2]:
import os, sys
from typing import Optional

import polars as pl
import pandas as pd

REPO_DATASET_PATH = "/kaggle/input/olympiadlevelmaths4llm/OlympiadLevelMaths4LLM"
sys.path.append(REPO_DATASET_PATH + "/src")

# --- Runtime / platform guards ---
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["TIKTOKEN_ENCODINGS_BASE"] = (
    "/kaggle/usr/lib/aimo3_packages_offline/tiktoken_encodings"
)

# =============================================================================
# V2 CONFIG — only env vars that AIMO3Config.from_env() actually reads
# Keep prompts compact so they do not overshadow short/easy problems.
# =============================================================================

# --- Prompts (override defaults from config.py) ---
os.environ["AIMO3_SYSTEM_PROMPT"] = (
    "Solve for the correct integer answer. "
    "Find the fastest exact method first. Do not guess from a pattern or partial argument. "
    "Prefer formulas, invariants, modular arithmetic, recurrences, and efficient counting. "
    "For counting, ordering, and divisibility problems, derive the exact expression and prime valuations exactly. "
    "Use Python early for exact integer arithmetic, small cases, and the final exact computation. "
    "Avoid floats, asymptotics, long proofs, and large brute force. "
    "Return only \\boxed{n} with 0 <= n <= 99999."
)

os.environ["AIMO3_TOOL_PROMPT"] = (
    "Use this stateful Python notebook to derive and verify an exact answer. "
    "Work with exact integer or rational arithmetic; avoid floats unless they are provably safe. "
    "For counting or divisibility, compute recurrences, products, factorizations, and p-adic valuations exactly. "
    "Start with tiny cases or symbolic simplification, then code the exact algorithm. "
    "Check complexity before loops, keep code short, print only decisive results, and always use print()."
)

os.environ["AIMO3_PREFERENCE_PROMPT"] = (
    "Find an exact method first. For counting or divisibility, derive the exact formula or valuation and verify with Python. Never guess from patterns."
 )


# --- Model / server ---
os.environ["AIMO3_MODEL_PATH"] = "/kaggle/input/gpt-oss-120b/transformers/default/1"
os.environ["AIMO3_SERVED_MODEL_NAME"] = "gpt-oss"
os.environ["AIMO3_REUSE_EXISTING_SERVER"] = "1"
os.environ["AIMO3_SERVER_TIMEOUT"] = "2000"
os.environ["AIMO3_REQUIRE_CUDA"] = "1"

# Cold-start speedup
os.environ["AIMO3_PRELOAD_MODEL_WEIGHTS"] = "1"
os.environ["AIMO3_PRELOAD_MODEL_WORKERS"] = "8"

# --- Display / tracing ---
os.environ["AIMO3_DISPLAY_CANDIDATES"] = "1"
os.environ["AIMO3_TRACE"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS"] = "0"
os.environ["AIMO3_TRACE_ENV"] = "1"
os.environ["AIMO3_TRACE_ENV_PACKAGES"] = "sympy,numpy,mpmath,jupyter_client,ortools,z3-solver"
os.environ["AIMO3_TRACE_INCLUDE_PROBLEM_TEXT"] = "0"

# --- Core decoding / capacity ---
os.environ["AIMO3_SEED"] = "42"
os.environ["AIMO3_SEARCH_TOKENS"] = "32"
os.environ["AIMO3_CONTEXT_TOKENS"] = "65536"
os.environ["AIMO3_BATCH_SIZE"] = "128"
os.environ["AIMO3_GPU_MEMORY_UTILIZATION"] = "0.96"

# --- Sandbox/tooling ---
os.environ["AIMO3_JUPYTER_TIMEOUT"] = "30"
os.environ["AIMO3_SANDBOX_TIMEOUT"] = "2"

# --- Time budgeting ---
os.environ["AIMO3_PROBLEMS_TOTAL"] = "50"
os.environ["AIMO3_NOTEBOOK_LIMIT"] = "17700"
os.environ["AIMO3_BASE_PROBLEM_TIMEOUT"] = "280"
os.environ["AIMO3_HIGH_PROBLEM_TIMEOUT"] = "1800"

# --- Attempt scheduling ---
os.environ["AIMO3_ATTEMPTS"] = "5"
os.environ["AIMO3_WORKERS"] = "8"
os.environ["AIMO3_TURNS"] = "128"
os.environ["AIMO3_EARLY_STOP"] = "3"
os.environ["AIMO3_EARLY_STOP_MIN_VERIFIED"] = "1"

# --- Extraction ---
os.environ["AIMO3_STRICT_FALLBACK_EXTRACTION"] = "1"

# --- Decoding knobs ---
os.environ["AIMO3_TEMPERATURE"] = "1.0"
os.environ["AIMO3_MIN_P"] = "0.02"
os.environ["AIMO3_TOP_P"] = "0.98"
os.environ["AIMO3_TOP_K"] = "-1"

# Keep the cheap verification phase on: it only triggers on weak consensus,
# which is exactly where popular-but-wrong answers tend to survive.
os.environ["AIMO3_VERIFY_PHASE_ENABLED"] = "0"
os.environ["AIMO3_VERIFY_DISABLE_GLOBALLY_IF_ALL_UNKNOWN"] = "1"
os.environ["AIMO3_VERIFY_ATTEMPTS_PER_CANDIDATE"] = "2"
os.environ["AIMO3_VERIFY_TOP_K_CANDIDATES"] = "2"
os.environ["AIMO3_VERIFY_TIMEOUT"] = "30"
os.environ["AIMO3_VERIFY_TEMPERATURE"] = "0.3"
os.environ["AIMO3_PYTHON_TOOL_VERIFY_REQUIRE_MARKER"] = "0"

# --- Time Management Approach ---
# - 'equal': use equal share of remaining time
# - 'base': use configured base_timeout_s for every problem
# - 'avg': use rolling average
# - 'cumulative': add carryover from previous problems
# - 'hybrid': take max(equal, avg, base) (default behavior)
os.environ["AIMO3_BUDGET_STRATEGY"] = "cumulative"
os.environ["AIMO3_BASE_TIMEOUT_S"] = ""  # unset to avoid overriding base_problem_timeout
os.environ["AIMO3_CARRYOVER_ENABLED"] = "1"
os.environ["AIMO3_CUMULATIVE_DISTRIBUTE"] = "0"

os.environ["AIMO3_WICKELGREN"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS_MAX_CHARS"] = "60000"

os.environ["AIMO3_FILTER_TO_VERIFIED_IF_ANY"] = "0"
os.environ["AIMO3_ENTROPY_WEIGHTING"] = "0"
os.environ["AIMO3_RANKING_STRATEGY"] = "votes_then_verified"

# --- CPU retriever (v2, v1-compatible env names) ---
# Set this path to your mounted KB dir containing concepts.json or concepts.pkl
os.environ["AIMO3_RETRIEVER_ENABLED"] = "0"
os.environ["AIMO3_RETRIEVER_KB_PATH"] = "/kaggle/input/olympiadlevelmaths4llmdb/OlympiadLevelMaths4LLMDB/knowledge_base"
os.environ["AIMO3_RETRIEVER_CPU_ONLY"] = "1"
os.environ["AIMO3_RETRIEVER_TOP_K"] = "5"
os.environ["AIMO3_RETRIEVER_MIN_SCORE"] = "0.08"
os.environ["AIMO3_RETRIEVER_INCLUDE_EXAMPLES"] = "1"
os.environ["AIMO3_RETRIEVER_INCLUDE_DEFINITIONS"] = "1"
os.environ["AIMO3_RETRIEVER_WARMUP_ON_INIT"] = "1"
os.environ["AIMO3_RETRIEVER_MODEL_PATH"] = "/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2"

os.environ["AIMO3_ADAPTIVE_BUDGET_FLEX_POOL_FRACTION"] = "0"

# --- Meta Learning ---
os.environ["AIMO3_META_LEARNING_ENABLED"] = "0"
os.environ["AIMO3_META_LEARNING_SIMILARITY_THRESHOLD"] = "0.3"

os.environ["AIMO3_Z3_TOOL_ENABLED"] = "0"

# Misc
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

os.environ["AIMO3_50_PROBLEMS_DATA_ENABLED"] = "1"

MODEL_PATH = os.getenv("AIMO3_MODEL_PATH", "")
print(
    f"Config: BASE_TIMEOUT={os.environ.get('AIMO3_BASE_PROBLEM_TIMEOUT')}s, HIGH_TIMEOUT={os.environ.get('AIMO3_HIGH_PROBLEM_TIMEOUT')}s"
)
print(
    f"Config: EARLY_STOP={os.environ.get('AIMO3_EARLY_STOP')}, MIN_VERIFIED={os.environ.get('AIMO3_EARLY_STOP_MIN_VERIFIED')}"
)
print(
    f"Config: ATTEMPTS={os.environ.get('AIMO3_ATTEMPTS')}, WORKERS={os.environ.get('AIMO3_WORKERS')}, TURNS={os.environ.get('AIMO3_TURNS')}"
)
print(
    f"Config: RANKING_STRATEGY={os.environ.get('AIMO3_RANKING_STRATEGY')}, FILTER_TO_VERIFIED_IF_ANY={os.environ.get('AIMO3_FILTER_TO_VERIFIED_IF_ANY')}"
)
print(
    f"Config: RETRIEVER_ENABLED={os.environ.get('AIMO3_RETRIEVER_ENABLED')}, RETRIEVER_KB_PATH={os.environ.get('AIMO3_RETRIEVER_KB_PATH')}"
)
MODEL_PATH

Config: BASE_TIMEOUT=280s, HIGH_TIMEOUT=1800s
Config: EARLY_STOP=3, MIN_VERIFIED=1
Config: ATTEMPTS=5, WORKERS=8, TURNS=128
Config: RANKING_STRATEGY=votes_then_verified, FILTER_TO_VERIFIED_IF_ANY=0
Config: RETRIEVER_ENABLED=0, RETRIEVER_KB_PATH=/kaggle/input/olympiadlevelmaths4llmdb/OlympiadLevelMaths4LLMDB/knowledge_base


'/kaggle/input/gpt-oss-120b/transformers/default/1'

In [3]:
# --- First-wave fast-answer mode ---
# Keep this smaller than the total attempt budget so at least a few full Python-backed
# attempts run before consensus can lock in.
os.environ["AIMO3_ANSWER_ONLY_ATTEMPTS"] = "2"
os.environ["AIMO3_ANSWER_ONLY_PROMPT"] = (
    "Find the integer with an exact method. Do not guess from a pattern or partial argument. "
    "Think silently. Do not explain. Return only \\boxed{number}."
 )

# Re-enable entropy-weighted tie-breaking to recover a confidence signal across attempts.
os.environ["AIMO3_ENTROPY_WEIGHTING"] = "1"

print(
    f"Config: ANSWER_ONLY_ATTEMPTS={os.environ.get('AIMO3_ANSWER_ONLY_ATTEMPTS')}, "
    f"ENTROPY_WEIGHTING={os.environ.get('AIMO3_ENTROPY_WEIGHTING')}"
 )

Config: ANSWER_ONLY_ATTEMPTS=2, ENTROPY_WEIGHTING=1


In [4]:
from olympiad_llm.aimo3.v2.cleanup import environ_setup_parallel, wait_all

# Start environment setup (uninstall + offline install + model warmup) ALL IN PARALLEL.
# This overlaps pip install with model cache warmup, saving ~30-60s on cold start.
ENVIRON_SETUP = environ_setup_parallel(warm_model=True, model_workers=8)
print("Started parallel environment setup:")
print("  - pip uninstall conflicts (background)")
print("  - pip install required packages (background)")
print("  - Model weight cache warmup (background)")
print("Will wait for completion lazily when the solver is first needed.")

Started parallel environment setup:
  - pip uninstall conflicts (background)
  - pip install required packages (background)
  - Model weight cache warmup (background)
Will wait for completion lazily when the solver is first needed.


In [5]:
import threading
from olympiad_llm.aimo3.v2.config import AIMO3Config
from olympiad_llm.aimo3.v2.runner import build_solver, run_kaggle_inference
from olympiad_llm.aimo3.v2.cleanup import wait_all

# IMPORTANT for Kaggle: don't block notebook execution on vLLM cold-start here.
# We'll build the solver lazily on the first predict() call.
cfg = AIMO3Config.from_env()
solver = None
_solver_lock = threading.Lock()

def get_solver():
    global solver
    if solver is not None:
        return solver
    with _solver_lock:
        if solver is not None:
            return solver
        # Wait for ALL parallel setup tasks to finish (pip + model warmup).
        if "ENVIRON_SETUP" in globals() and isinstance(ENVIRON_SETUP, dict):
            wait_all(ENVIRON_SETUP, timeout=180)
            print("✓ Parallel setup complete (pip + model cache warmup)")
        solver = build_solver(cfg)
        return solver

print("Lazy solver configured. Inference server can start now.")

Lazy solver configured. Inference server can start now.


In [6]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions, ground_truth
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    
    # Build solver only when needed (keeps Kaggle inference server startup fast).
    s = get_solver()
    final_answer = s.solve_problem(question_text)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available (local runs only).
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [7]:
from olympiad_llm.aimo3.prepare import prepare_reference_csv

In [8]:
# Load ground truth only for local testing (avoid delaying server start in competition reruns).
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    ground_truth = {}
elif int(os.environ.get('AIMO3_50_PROBLEMS_DATA_ENABLED')):
    df = pd.read_csv('/kaggle/input/olympiadlevelmaths4llm/50problems.csv')
    df.insert(0, 'id', range(1, len(df) + 1))
    df.rename(columns={'Problem': 'problem', 'Answer': 'answer'}, inplace=True)
    print(df.head())
    df.to_csv('50problems.csv', index=False)
    ground_truth, _ = prepare_reference_csv(
        "50problems.csv",
    )
else:
    ground_truth, _ = prepare_reference_csv(
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv",
        # "/kaggle/input/olympiadlevelmaths4llmdb/inmo_1986.csv",
        # problem_ids=["dd7f5e", "86e8e5"],
        # problem_ids=["86e8e5"],
    )

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

   id                                            problem  answer
0   1  Let $ABC$ be an acute-angled triangle with int...     336
1   2  Define a function $f \colon \mathbb{Z}_{\geq 1...   32951
2   3  A tournament is held with $2^{20}$ runners eac...   21818
3   4  Ken writes a positive integer $n$ on a blackbo...   32193
4   5  Let triangle $ABC$ be $n$-tastic if $BD = F_n,...   57447


In [9]:
inference_server = run_kaggle_inference(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(("reference.csv",))

------
ID: 41
✓ Parallel setup complete (pip + model cache warmup)

Problem: A region is formed by three unit squares in an L-shape. Two points are chosen randomly. Find $m+n$ if the probability their midpoint is inside the region is $m/n$.

Budget: 354.00s | [Budget] 0/50 done | Remaining: 17700s | Flex: 0s/0s | Avg: 280s | Next: 354s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,35,True,4,0,0,17446,0.671732,...t/N\n\nprint(trial(2000000))\nanalysis0.944...,None
1,2,35,False,0,0,0,7201,0.654305,"...rior and possibly on boundary? Usually ""ins...",None
2,3,35,True,1,0,0,7373,0.689026,...on equals 1 - probability that midpoint lan...,None



Final Answer: 35 (votes=3, verified=2)

Answer: 35 | Ground Truth: 35 | ✅
📊 Running Accuracy: 1/1 (100.0%)
------

------
ID: 7

Problem: Alice and Bob each hold some sweets. Alice says: If we added our sweets to our positive integer age, my answer would be double yours. If we took the product, my answer would be four times yours. Bob says: Give me five sweets and then both our sum and product would be equal. What is the product of Alice and Bob's ages?

Budget: 533.39s | [Budget] 1/50 done | Remaining: 17525s | Flex: 0s/0s | Avg: 175s | Next: 358s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,50,False,0,0,0,3201,0.540702,...oduct: Alice product = 10*10=100. Bob produ...,None
1,2,50,False,0,0,0,3801,0.595083,"...y to X & N? Actually X & N gave A=10, B=5 a...",None
2,5,50,True,2,0,0,2759,0.635234,...ider whether any other solutions exist with...,None



Final Answer: 50 (votes=3, verified=1)

Answer: 50 | Ground Truth: 50 | ✅
📊 Running Accuracy: 2/2 (100.0%)
------

------
ID: 29

Problem: Find the number of rectangles formed inside a regular 12-gon where each side lies on either a side or a diagonal of the dodecagon.

Budget: 841.76s | [Budget] 2/50 done | Remaining: 17480s | Flex: 0s/0s | Avg: 110s | Next: 364s | Extensions: 0



...


Final Answer: 211 (votes=3, verified=3)

Answer: 211 | Ground Truth: 211 | ✅
📊 Running Accuracy: 41/48 (85.4%)
------

------
ID: 22

Problem: Find the number of ways to place a digit in each cell of a $2 \times 3$ grid so the sum of the two 3-digit numbers reading left to right is 999, and the sum of the three 2-digit numbers reading top to bottom is 99.

Budget: 1800.00s | [Budget] 48/50 done | Remaining: 9919s | Flex: 0s/0s | Avg: 118s | Next: 1800s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,45,True,2,0,0,5328,0.644496,...rrectly forced a+d=0; correct is a+d=9. Goo...,None
1,2,21,True,1,0,0,4125,0.672536,"...[VERIFICATION NOTICE] TOOL_OUTPUT_INVALID""....",None
2,3,21,True,2,0,0,4312,0.670987,"...b,c,d each must be 1-9, e,f can be 0-9. Our...",None
3,5,21,True,1,0,0,4373,0.718791,...ced. So count 21.\n\nThus answer is 21.\n\n...,None



Final Answer: 21 (votes=3, verified=3)

Answer: 21 | Ground Truth: 45 | ❌
📊 Running Accuracy: 41/49 (83.7%)
------

------
ID: 30

Problem: Five men and nine women stand in a circle. The probability that every man stands diametrically opposite a woman is $m/n$. Find $m+n$.

Budget: 1800.00s | [Budget] 49/50 done | Remaining: 9852s | Flex: 0s/0s | Avg: 122s | Next: 1800s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,191,True,6,0,0,3266,0.847493,"...seems correct.\n\nThus m=48,n=143, m+n=191....",None
1,2,191,True,11,0,0,2691,0.814883,...reat only rotations.\n\nNow let's reason mo...,None
2,4,191,True,6,0,0,3135,0.757097,...ce final answer.assistantfinalThe 14 positi...,None



Final Answer: 191 (votes=3, verified=3)

Answer: 191 | Ground Truth: 191 | ✅
📊 Running Accuracy: 42/50 (84.0%)
------

